<a href="https://colab.research.google.com/github/Subparplatinum/Moss-Species-Classifier/blob/main/moss_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moss classification model creation

# Download dataset

In [40]:
import kagglehub
path = kagglehub.dataset_download("alexanderuzhinskiy/moss-species-classification-dataset")


Using Colab cache for faster access to the 'moss-species-classification-dataset' dataset.


# Import and preprocess data

In [41]:
import os
import torch
import numpy as np
from glob import glob
from PIL import Image
from torch.utils.data import random_split, Dataset, DataLoader
from torchvision import transforms as T

torch.manual_seed(2025)

class CustomDataset(Dataset):
    def __init__(self, root, transformations=None, im_files=[".png", ".jpg"]):

        self.transformations = transformations
        self.im_paths = glob(f"{root}/*/*{[im_file for im_file in im_files]}")

        self.cls_names, self.cls_counts = {}, {}
        count = 0
        for im_path in self.im_paths:
            class_name = self.get_class(im_path)
            if class_name not in self.cls_names:
                self.cls_names[class_name] = count
                self.cls_counts[class_name] = 1
                count += 1
            else:
                self.cls_counts[class_name] += 1

    def get_class(self, path): return os.path.dirname(path).split("/")[-1]

    def __len__(self): return len(self.im_paths)

    def __getitem__(self, idx):

        im_path = self.im_paths[idx]
        im = Image.open(im_path).convert("RGB")
        gt = self.cls_names[self.get_class(im_path)]

        if self.transformations is not None:
            im = self.transformations(im)

        return im, gt

    @classmethod
    def get_dls(cls, root, transformations, bs, split=[0.9, 0.05, 0.05], ns=4):

        ds = cls(root=root, transformations=transformations)

        total_len = len(ds)
        tr_len = int(total_len * split[0])
        vl_len = int(total_len * split[1])
        ts_len = total_len - (tr_len + vl_len)

        tr_ds, vl_ds, ts_ds = random_split(ds, [tr_len, vl_len, ts_len])

        tr_dl = DataLoader(tr_ds, batch_size=bs, shuffle=True, num_workers=ns)
        val_dl = DataLoader(vl_ds, batch_size=bs, shuffle=False, num_workers=ns)
        ts_dl = DataLoader(ts_ds, batch_size=1, shuffle=False, num_workers=ns)

        return tr_dl, val_dl, ts_dl, ds.cls_names, ds.cls_counts

root = "/kaggle/input/moss-species-classification-dataset/mosses"
mean, std, im_size, bs = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 224, 16
tfs = T.Compose([T.Resize((im_size, im_size)), T.ToTensor(), T.Normalize(mean=mean, std=std)])

trainloader, testloader, ts_dl, classes, cls_counts = CustomDataset.get_dls(root=root, transformations=tfs, bs=bs)

print(len(trainloader)); print(len(testloader)); print(len(ts_dl)); print(classes)

34
2
31
{'hypnum_cupressiforme': 0, 'pseudoscleropodium_purum': 1, 'pleurozium_schreberi': 2, 'hylocomium_splendens': 3, 'abietinella_abietina': 4}


# Define Network

In [42]:
import torch.nn as nn
import torch.nn.functional as F

# Define the device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        # Create a dummy input to pass through the conv layers to determine output size

        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 224, 224) # Files are 256 x 256, but 224x224 is good practice
            x = self.pool(F.relu(self.conv1(dummy_input)))
            x = self.pool(F.relu(self.conv2(x)))
            self._to_linear = torch.flatten(x, 1).shape[1]

        self.fc1 = nn.Linear(self._to_linear, 128)
        self.fc2 = nn.Linear(128, 32)
        self.fc3 = nn.Linear(32, len(classes))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()
net.to(device) # Move the model to the GPU

import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

cpu


# Train Model

In [43]:
for epoch in range(10):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data[0].to(device), data[1].to(device) # Move inputs and labels to GPU

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:    # print every 100 mini-batches
          print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
          running_loss = 0.0

print('Finished Training')

torch.save(net.state_dict(), "/root/moss_classifier.pth")

Finished Training


# Test

In [44]:
net = Net()
net.load_state_dict(torch.load("/root/moss_classifier.pth", weights_only=True))

correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device) # Move images and labels to GPU
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the {total} test images: {100 * correct // total} %')

Accuracy of the model on the 29 test images: 37 %
